In [ ]:
"""
Flight Price Prediction using Linear Regression
Dataset: https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction
Use the file: Clean_Dataset.csv  (place it in the same folder as this script)

Install requirements:
    pip install pandas numpy matplotlib seaborn scikit-learn
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------------
df = pd.read_csv("Clean_Dataset.csv")
print("Shape:", df.shape)
print(df.head())
print("\nMissing values:\n", df.isnull().sum())

# ------------------------------------------------------------------
# 2. CLEANING
# ------------------------------------------------------------------
# Drop the unnamed index column and 'flight' (flight code, too many unique values)
df = df.drop(columns=[c for c in ["Unnamed: 0", "flight"] if c in df.columns])
df = df.drop_duplicates()

# ------------------------------------------------------------------
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ------------------------------------------------------------------
sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x="airline", y="price", data=df, ax=axes[0, 0])
axes[0, 0].set_title("Price by Airline")
axes[0, 0].tick_params(axis="x", rotation=30)

sns.boxplot(x="class", y="price", data=df, ax=axes[0, 1])
axes[0, 1].set_title("Price by Class")

sns.boxplot(x="stops", y="price", data=df, ax=axes[1, 0])
axes[1, 0].set_title("Price by Number of Stops")

sample = df.sample(min(5000, len(df)), random_state=42)
sns.scatterplot(x="days_left", y="price", data=sample, alpha=0.3, ax=axes[1, 1])
axes[1, 1].set_title("Price vs Days Left Before Departure")

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=150)
plt.show()

# ------------------------------------------------------------------
# 4. FEATURE ENGINEERING
# ------------------------------------------------------------------
# 'stops' is ordered, so map it to numbers
df["stops"] = df["stops"].map({"zero": 0, "one": 1, "two_or_more": 2})

# 'class' is binary
df["class"] = df["class"].map({"Economy": 0, "Business": 1})

# One-hot encode the remaining categorical columns
categorical_cols = ["airline", "source_city", "departure_time",
                    "arrival_time", "destination_city"]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# ------------------------------------------------------------------
# 5. TRAIN / TEST SPLIT
# ------------------------------------------------------------------
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\nTrain size:", X_train.shape, " Test size:", X_test.shape)

# ------------------------------------------------------------------
# 6. TRAIN LINEAR REGRESSION MODEL
# ------------------------------------------------------------------
model = LinearRegression()
model.fit(X_train, y_train)

# ------------------------------------------------------------------
# 7. EVALUATE
# ------------------------------------------------------------------
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== MODEL PERFORMANCE =====")
print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R2   : {r2:.4f}")

# Actual vs Predicted plot
plt.figure(figsize=(7, 7))
plt.scatter(y_test[:3000], y_pred[:3000], alpha=0.3)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Flight Prices")
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150)
plt.show()

# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=60, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.savefig("residuals.png", dpi=150)
plt.show()

# ------------------------------------------------------------------
# 8. FEATURE IMPORTANCE (coefficients)
# ------------------------------------------------------------------
coef_df = pd.DataFrame({"feature": X.columns, "coefficient": model.coef_})
coef_df["abs_coef"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False).head(15)

plt.figure(figsize=(9, 6))
sns.barplot(x="coefficient", y="feature", data=coef_df)
plt.title("Top 15 Features Affecting Price")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()

print("\nTop features:\n", coef_df[["feature", "coefficient"]])

# ------------------------------------------------------------------
# 9. PREDICT FOR A NEW FLIGHT
# ------------------------------------------------------------------
def predict_price(airline, source_city, departure_time, stops,
                  arrival_time, destination_city, travel_class,
                  duration, days_left):
    """Return predicted ticket price for a single flight."""
    row = {col: 0 for col in X.columns}
    row["stops"] = {"zero": 0, "one": 1, "two_or_more": 2}[stops]
    row["class"] = 1 if travel_class == "Business" else 0
    row["duration"] = duration
    row["days_left"] = days_left

    for col in [f"airline_{airline}", f"source_city_{source_city}",
                f"departure_time_{departure_time}",
                f"arrival_time_{arrival_time}",
                f"destination_city_{destination_city}"]:
        if col in row:          # the dropped baseline category stays all zeros
            row[col] = 1

    return model.predict(pd.DataFrame([row]))[0]


example = predict_price(
    airline="Vistara", source_city="Delhi", departure_time="Morning",
    stops="zero", arrival_time="Evening", destination_city="Mumbai",
    travel_class="Economy", duration=2.5, days_left=20,
)
print(f"\nPredicted price for sample flight: Rs. {example:,.0f}")
